# Export dashboard data

Refreshes the CSVs behind the static Delivery SLA dashboard published on GitHub Pages (`docs/dashboard`).

**What it does**
- Runs one query per dashboard dataset against Gold (plus Silver and Bronze where Gold doesn't carry the column)
- Writes one CSV per dataset to the landing volume, under `exports/dashboard/`
- Prints row counts so you can check nothing came back empty

**Good to know**
- Read-only on the platform tables. The only write is the CSV export.
- `distance_bands.csv` is not exported: it needs the Olist geolocation table, which the platform doesn't ingest yet.
- The simulated datasets (`sim_*.csv`) are not exported either; they are illustrative and live only in the repo.
- To publish: download the CSVs, replace the files in `docs/dashboard/data/`, and open a PR.


In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "stg", "prd"], "Environment")
dbutils.widgets.text("catalog", "de_databricks_bp_wave_5", "Catalog")

ENV     = dbutils.widgets.get("env")
CATALOG = dbutils.widgets.get("catalog")
SCHEMAS = {layer: f"{CATALOG}.olist_{ENV}_{layer}" for layer in ("gold", "silver", "bronze")}
OUT_DIR = f"/Volumes/{CATALOG}/olist_dev_landing/olist_raw/exports/dashboard"

print(SCHEMAS)
print("Writing to", OUT_DIR)


In [0]:
# Olist publishes Portuguese category names; the dashboard shows English ones.
# Mapping from the dataset's own product_category_name_translation.csv.
CATEGORY_TRANSLATION = [
    ("beleza_saude", "health_beauty"),
    ("informatica_acessorios", "computers_accessories"),
    ("automotivo", "auto"),
    ("cama_mesa_banho", "bed_bath_table"),
    ("moveis_decoracao", "furniture_decor"),
    ("esporte_lazer", "sports_leisure"),
    ("perfumaria", "perfumery"),
    ("utilidades_domesticas", "housewares"),
    ("telefonia", "telephony"),
    ("relogios_presentes", "watches_gifts"),
    ("alimentos_bebidas", "food_drink"),
    ("bebes", "baby"),
    ("papelaria", "stationery"),
    ("tablets_impressao_imagem", "tablets_printing_image"),
    ("brinquedos", "toys"),
    ("telefonia_fixa", "fixed_telephony"),
    ("ferramentas_jardim", "garden_tools"),
    ("fashion_bolsas_e_acessorios", "fashion_bags_accessories"),
    ("eletroportateis", "small_appliances"),
    ("consoles_games", "consoles_games"),
    ("audio", "audio"),
    ("fashion_calcados", "fashion_shoes"),
    ("cool_stuff", "cool_stuff"),
    ("malas_acessorios", "luggage_accessories"),
    ("climatizacao", "air_conditioning"),
    ("construcao_ferramentas_construcao", "construction_tools_construction"),
    ("moveis_cozinha_area_de_servico_jantar_e_jardim", "kitchen_dining_laundry_garden_furniture"),
    ("construcao_ferramentas_jardim", "costruction_tools_garden"),
    ("fashion_roupa_masculina", "fashion_male_clothing"),
    ("pet_shop", "pet_shop"),
    ("moveis_escritorio", "office_furniture"),
    ("market_place", "market_place"),
    ("eletronicos", "electronics"),
    ("eletrodomesticos", "home_appliances"),
    ("artigos_de_festas", "party_supplies"),
    ("casa_conforto", "home_confort"),
    ("construcao_ferramentas_ferramentas", "costruction_tools_tools"),
    ("agro_industria_e_comercio", "agro_industry_and_commerce"),
    ("moveis_colchao_e_estofado", "furniture_mattress_and_upholstery"),
    ("livros_tecnicos", "books_technical"),
    ("casa_construcao", "home_construction"),
    ("instrumentos_musicais", "musical_instruments"),
    ("moveis_sala", "furniture_living_room"),
    ("construcao_ferramentas_iluminacao", "construction_tools_lights"),
    ("industria_comercio_e_negocios", "industry_commerce_and_business"),
    ("alimentos", "food"),
    ("artes", "art"),
    ("moveis_quarto", "furniture_bedroom"),
    ("livros_interesse_geral", "books_general_interest"),
    ("construcao_ferramentas_seguranca", "construction_tools_safety"),
    ("fashion_underwear_e_moda_praia", "fashion_underwear_beach"),
    ("fashion_esporte", "fashion_sport"),
    ("sinalizacao_e_seguranca", "signaling_and_security"),
    ("pcs", "computers"),
    ("artigos_de_natal", "christmas_supplies"),
    ("fashion_roupa_feminina", "fashio_female_clothing"),
    ("eletrodomesticos_2", "home_appliances_2"),
    ("livros_importados", "books_imported"),
    ("bebidas", "drinks"),
    ("cine_foto", "cine_photo"),
    ("la_cuisine", "la_cuisine"),
    ("musica", "music"),
    ("casa_conforto_2", "home_comfort_2"),
    ("portateis_casa_forno_e_cafe", "small_appliances_home_oven_and_coffee"),
    ("cds_dvds_musicais", "cds_dvds_musicals"),
    ("dvds_blu_ray", "dvds_blu_ray"),
    ("flores", "flowers"),
    ("artes_e_artesanato", "arts_and_craftmanship"),
    ("fraldas_higiene", "diapers_and_hygiene"),
    ("fashion_roupa_infanto_juvenil", "fashion_childrens_clothes"),
    ("seguros_e_servicos", "security_and_services")
]
spark.createDataFrame(CATEGORY_TRANSLATION, "portuguese string, english string") \
     .createOrReplaceTempView("category_translation")


In [0]:
import os
from dashboard_queries import QUERIES, query

os.makedirs(OUT_DIR, exist_ok=True)

def tidy(pdf):
    # Whole-number columns come back as decimals or floats; write them as integers.
    for col in pdf.columns:
        s = pdf[col]
        if s.dtype == object or str(s.dtype).startswith("bool"):
            continue
        s = s.astype("float64")
        if s.notna().all() and (s % 1 == 0).all():
            pdf[col] = s.astype("int64")
        else:
            pdf[col] = s
    return pdf

counts = {}
for name in QUERIES:
    pdf = tidy(spark.sql(query(name, **SCHEMAS)).toPandas())
    pdf.to_csv(f"{OUT_DIR}/{name}.csv", index=False)
    counts[name] = len(pdf)

display(spark.createDataFrame(list(counts.items()), "dataset string, rows int"))
